# ML Network Intrusion Detector

**Project:** Machine Learning-Based Network Intrusion Detection  
**Dataset:** UNSW-NB15  
**Objective:** Build and evaluate machine-learning classifiers that distinguish normal network traffic from malicious traffic.

This notebook:
1. Loads the official UNSW-NB15 training and testing partitions.
2. Performs data validation and exploratory analysis.
3. Removes identifiers and target-leaking fields.
4. Preprocesses numerical and categorical network-flow features.
5. Trains a Logistic Regression baseline and a Random Forest classifier.
6. Evaluates both models using accuracy, precision, recall, F1-score, confusion matrix, and ROC-AUC.
7. Saves the best model and preprocessing pipeline for reuse.

> **Important:** Download the UNSW-NB15 training/testing CSV files before running the notebook. The notebook does not fabricate results when the dataset is unavailable.

## 1. Dataset

UNSW-NB15 is a network intrusion-detection benchmark containing normal traffic and nine attack categories. The official UNSW page provides a predefined training set of 175,341 records and a testing set of 82,332 records.

Dataset source: https://research.unsw.edu.au/projects/unsw-nb15-dataset

Expected files:
- `UNSW_NB15_training-set.csv`
- `UNSW_NB15_testing-set.csv`

The binary target is `label`, where `0` represents normal traffic and `1` represents an attack. `attack_cat` is retained for analysis but is **not** used as a predictive feature because it directly describes the attack class and would cause target leakage.

In [ ]:
# Install requirements separately if needed:
# pip install -r requirements.txt

from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay, RocCurveDisplay
)
import joblib

pd.set_option("display.max_columns", 100)

DATA_DIR = Path("data")

TRAIN_FILE = DATA_DIR / "UNSW_NB15_training-set.csv"
TEST_FILE = DATA_DIR / "UNSW_NB15_testing-set.csv"

print("Training file:", TRAIN_FILE)
print("Testing file :", TEST_FILE)


In [ ]:
def load_dataset(train_path=TRAIN_FILE, test_path=TEST_FILE):
    if not train_path.exists() or not test_path.exists():
        raise FileNotFoundError(
            "Dataset files were not found. Place UNSW_NB15_training-set.csv and "
            "UNSW_NB15_testing-set.csv inside a folder named 'data'. "
            "See README.md for the official dataset link."
        )

    train = pd.read_csv(train_path)
    test = pd.read_csv(test_path)

    # Normalize column names.
    train.columns = [str(c).strip() for c in train.columns]
    test.columns = [str(c).strip() for c in test.columns]

    return train, test

train_df, test_df = load_dataset()

print("Training shape:", train_df.shape)
print("Testing shape :", test_df.shape)
display(train_df.head())


In [ ]:
# Basic data validation
print("Missing values in training data:")
display(train_df.isna().sum().sort_values(ascending=False).head(15))

print("\nTarget distribution:")
display(train_df["label"].value_counts().rename(index={0: "Normal", 1: "Attack"}))

if "attack_cat" in train_df.columns:
    print("\nAttack-category distribution:")
    display(train_df["attack_cat"].fillna("Normal/Unknown").value_counts())


In [ ]:
# Visualize the binary class distribution.
class_counts = train_df["label"].value_counts().sort_index()

plt.figure(figsize=(6, 4))
plt.bar(["Normal (0)", "Attack (1)"], class_counts.values)
plt.title("UNSW-NB15 Training Class Distribution")
plt.ylabel("Number of records")
plt.tight_layout()
plt.show()


## 2. Preprocessing and Feature Selection

The following fields are excluded from model training:

- `label`: target variable.
- `attack_cat`: target-derived attack category; using it would leak the answer.
- `id`: row identifier.
- `srcip`, `dstip`: high-cardinality endpoint identifiers that can encourage memorization rather than generalizable traffic-pattern learning.
- `stime`, `ltime`: timestamps that may encode dataset-specific ordering/time effects.

Categorical fields are one-hot encoded. Numerical fields are median-imputed and standardized for the linear baseline. The preprocessing is placed inside a scikit-learn `Pipeline`/`ColumnTransformer` so that the same transformations are applied consistently during training and testing.

In [ ]:
TARGET = "label"

DROP_COLUMNS = [
    "id", "attack_cat", "srcip", "dstip", "stime", "ltime"
]

# Keep only columns that actually exist in the supplied dataset.
drop_existing = [c for c in DROP_COLUMNS if c in train_df.columns]

X_train = train_df.drop(columns=[TARGET] + drop_existing, errors="ignore")
y_train = train_df[TARGET].astype(int)

X_test = test_df.drop(columns=[TARGET] + drop_existing, errors="ignore")
y_test = test_df[TARGET].astype(int)

# Align columns in case the CSV files differ slightly.
X_test = X_test.reindex(columns=X_train.columns, fill_value=np.nan)

categorical_features = X_train.select_dtypes(include=["object", "category"]).columns.tolist()
numeric_features = [c for c in X_train.columns if c not in categorical_features]

print("Features used:", len(X_train.columns))
print("Numerical features:", len(numeric_features))
print("Categorical features:", categorical_features)


In [ ]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ],
    remainder="drop"
)

print("Preprocessing pipeline created.")


## 3. Model Training

Two models are compared:

**Logistic Regression** — interpretable and useful as a strong linear baseline.

**Random Forest** — a nonlinear ensemble model that can capture interactions among network-flow features and is relatively robust to mixed feature behavior.

For intrusion detection, **F1-score** is emphasized because it balances precision and recall. Recall is particularly important because false negatives correspond to malicious traffic that the detector fails to identify.

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=500,
        class_weight="balanced",
        solver="liblinear",
        random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=150,
        max_depth=20,
        min_samples_leaf=2,
        class_weight="balanced_subsample",
        n_jobs=-1,
        random_state=42
    )
}

pipelines = {
    name: Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ])
    for name, model in models.items()
}

results = []

for name, pipeline in pipelines.items():
    print(f"Training {name}...")
    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_test)
    y_prob = pipeline.predict_proba(X_test)[:, 1]

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1": f1_score(y_test, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, y_prob)
    })

results_df = pd.DataFrame(results).sort_values("F1", ascending=False)
display(results_df)


In [ ]:
# Detailed evaluation of the best-performing model by F1.
best_name = results_df.iloc[0]["Model"]
best_pipeline = pipelines[best_name]

best_pred = best_pipeline.predict(X_test)
best_prob = best_pipeline.predict_proba(X_test)[:, 1]

print("Best model:", best_name)
print("\nClassification report:")
print(classification_report(
    y_test,
    best_pred,
    target_names=["Normal", "Attack"],
    digits=4,
    zero_division=0
))

cm = confusion_matrix(y_test, best_pred)
print("Confusion matrix:")
display(pd.DataFrame(
    cm,
    index=["Actual Normal", "Actual Attack"],
    columns=["Predicted Normal", "Predicted Attack"]
))


In [ ]:
# Confusion matrix
ConfusionMatrixDisplay.from_predictions(
    y_test,
    best_pred,
    display_labels=["Normal", "Attack"]
)
plt.title(f"Confusion Matrix — {best_name}")
plt.tight_layout()
plt.show()

# ROC curve
RocCurveDisplay.from_predictions(y_test, best_prob)
plt.title(f"ROC Curve — {best_name}")
plt.tight_layout()
plt.show()


## 4. Feature Importance

For the Random Forest model, feature importance can provide an interpretable indication of which transformed network-flow attributes contributed most to classification.

Because preprocessing may expand categorical features into multiple one-hot columns, the code below retrieves the transformed feature names before ranking them.

In [ ]:
if "Random Forest" in pipelines:
    rf_pipeline = pipelines["Random Forest"]
    rf_model = rf_pipeline.named_steps["model"]
    rf_preprocessor = rf_pipeline.named_steps["preprocessor"]

    feature_names = rf_preprocessor.get_feature_names_out()
    importances = rf_model.feature_importances_

    importance_df = (
        pd.DataFrame({
            "Feature": feature_names,
            "Importance": importances
        })
        .sort_values("Importance", ascending=False)
        .head(20)
    )

    display(importance_df)

    plt.figure(figsize=(8, 7))
    plt.barh(
        importance_df["Feature"].iloc[::-1],
        importance_df["Importance"].iloc[::-1]
    )
    plt.title("Top 20 Random Forest Feature Importances")
    plt.xlabel("Importance")
    plt.tight_layout()
    plt.show()


In [ ]:
# Save the best complete pipeline.
MODEL_DIR = Path("artifacts")
MODEL_DIR.mkdir(exist_ok=True)

model_path = MODEL_DIR / "best_intrusion_detector.joblib"
joblib.dump(best_pipeline, model_path)

# Save the model comparison table for the report/submission folder.
results_df.to_csv(MODEL_DIR / "model_results.csv", index=False)

print("Saved:", model_path)
print("Saved:", MODEL_DIR / "model_results.csv")


## 5. Example Inference

The saved pipeline includes preprocessing and the classifier, so a new raw record can be passed through the same transformations.

Replace the example with a row from a compatible UNSW-NB15 test record when demonstrating inference.

In [ ]:
# Example: classify the first test record.
# This uses the raw feature columns expected by the trained pipeline.
sample = X_test.iloc[[0]]

prediction = best_pipeline.predict(sample)[0]
probability = best_pipeline.predict_proba(sample)[0, 1]

print("Prediction:", "Attack" if prediction == 1 else "Normal")
print(f"Attack probability: {probability:.4f}")


## 6. Conclusions

This project demonstrates an end-to-end machine-learning workflow for network intrusion detection:

- A public cybersecurity benchmark is used for supervised learning.
- Target leakage is controlled by excluding `attack_cat` from predictive features.
- A preprocessing pipeline handles missing values, categorical variables, and numerical scaling.
- Logistic Regression provides a baseline while Random Forest provides a nonlinear comparison.
- Precision, recall, F1-score, ROC-AUC, and a confusion matrix are used instead of relying only on accuracy.
- The final pipeline is serialized with Joblib so it can be integrated into a larger monitoring or API system.

### Limitations

1. The model is trained on a benchmark dataset and may not generalize directly to unseen enterprise traffic.
2. Removing IP addresses and timestamps improves generalization but also removes potentially useful operational context.
3. Dataset class distributions and attack patterns may differ from current real-world threats.
4. A production IDS would require continuous retraining, drift monitoring, threshold tuning, and validation against live traffic.

### Ethical and Security Note

This project is intended for defensive cybersecurity research and academic learning. It analyzes labeled network-flow data and does not perform attacks against live systems.